In [ ]:
import pandas as pd

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_csv("../data/churn.csv")

print("Columns:", df.columns)

# -----------------------------
# CLEANING
# -----------------------------

# Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Drop missing values
df.dropna(inplace=True)

# -----------------------------
# FEATURE ENGINEERING
# -----------------------------

# 1. Average charge per month
df['AvgCharges'] = df['TotalCharges'] / (df['tenure'] + 1)

# 2. Customer tenure group
def tenure_group(x):
    if x <= 12:
        return "New"
    elif x <= 36:
        return "Mid"
    else:
        return "Old"

df['TenureGroup'] = df['tenure'].apply(tenure_group)

# Convert categorical to numeric
df = pd.get_dummies(df, columns=['TenureGroup'], drop_first=True)

# -----------------------------
# FEATURES & TARGET
# -----------------------------
X = df.drop(columns=["Churn"])
y = df["Churn"]

# -----------------------------
# SCALING
# -----------------------------
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# -----------------------------
# TRAIN TEST SPLIT
# -----------------------------
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# -----------------------------
# MODEL
# -----------------------------
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# -----------------------------
# PREDICTION
# -----------------------------
y_pred = model.predict(X_test)

# -----------------------------
# RESULT
# -----------------------------
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))